## Documento de Requerimientos del Sistema (DRS) - Fase 4: Arquitectura y API Robusta

Llegamos a la **Fase 4**, el cierre de la Unidad 2. Aquí es donde el sistema deja de ser un simple CRUD y se convierte en un motor transaccional aplicando concurrencia y patrones de diseño avanzados.

El objetivo es procesar la compra que el estudiante envía desde `Checkout.tsx`, asegurando que la base de datos mantenga propiedades ACID (si algo falla, se revierte todo) y utilizando *Goroutines* para calcular las comisiones de la red multinivel en segundo plano sin bloquear la respuesta al usuario final.

### Instrucciones de Ejecución para la IA (Fase 4)

Copia y pega este último bloque en tu entorno de desarrollo:

```markdown
> **Objetivo:** Aplicar inyección de dependencias para desacoplar la arquitectura y construir un microservicio seguro (`/api/checkout`) que maneje transacciones ACID en PostgreSQL y utilice concurrencia (Goroutines) para procesos asíncronos.
> 
> **Paso 1: Inyección de Dependencias (Tema 4)**
> Refactoriza la capa de controladores y repositorios. En lugar de usar funciones sueltas y variables globales, crea estructuras (`struct`) con interfaces inyectadas. Por ejemplo, `type CheckoutController struct { repo DBRepository, cache CacheRepository }`. Actualiza `routes.go` y `main.go` para instanciar e inyectar estas dependencias al arrancar el servidor.
> 
> **Paso 2: Microservicio de Simulación de Transacciones (Tema 4)**
> Crea un nuevo endpoint `POST /api/checkout` que reciba el JSON enviado por el componente `Checkout.tsx` del frontend (datos del cliente, método de pago y arreglo de items del carrito).
> 
> **Paso 3: Transacciones ACID (PostgreSQL)**
> En el repositorio, la función que guarda el pedido debe iniciar una transacción SQL (`BEGIN`). 
> 1. Inserta el registro principal del pedido en una tabla `pedidos`.
> 2. Inserta el detalle en `pedido_items` restando el stock simulado.
> 3. Si cualquier paso falla (ej. base de datos caída o validación incorrecta), ejecuta un `ROLLBACK` y retorna un HTTP 500 o 400. Si todo es exitoso, ejecuta `COMMIT`.
> 
> **Paso 4: Concurrencia y Misión Crítica (Tema 1)**
> El cálculo de comisiones para la red multinivel no debe hacer esperar al cliente en la pantalla de pago. Si el `COMMIT` del Paso 3 es exitoso, lanza una **Goroutine** (`go calcularComisiones(pedido.Total, usuario.ID)`) que actualice los saldos de los referidos ascendentes (niveles 1, 2 y 3) de forma asíncrona en PostgreSQL. 
> 
> **Paso 5: Invalidación de Caché**
> Al finalizar la compra y actualizar las comisiones, asegúrate de eliminar la clave de caché en Redis (`red:tree:{email_usuario}`) para que la próxima vez que el usuario entre al Dashboard, obtenga los KPIs actualizados.

```

### ¿Cómo ven pantalla con tus estudiantes?

1. **Flujo Feliz:** Entra a la aplicación React como cliente, agrega productos al carrito y ve a "Proceder al Pago". Llena el formulario y confirma. El servidor Go procesará la orden e inmediatamente verás la pantalla verde de `Confirmacion.tsx`.


2. **El poder de la Concurrencia:** Inicia sesión ahora como Administrador y vPodras vershboard (`/`). Tus estudiantes verán que los KPIs de "Comisiones del Mes" y "Ventas de la Red" han subido mágicamente. Explicarás que esto lo hizo la *Goroutine* en segundo plano mientras el cliente ya estaba viendo su recibo.


3. **Simulación de Falla (ACID):** Provoca un error intencional (por ejemplo, desconecta temporalmente un cable de red o altera un dato para forzar un error SQL a mitad de la inserción del detalle). Al intentar comprar, la pantalla de React no avanzará; mostrará la alerta puedenvisar la be de datos, los estudiantes comprobarán que no se creó un "pedido fantasma" ni se pagaron comisiones inválidas gracias al `ROLLBACK`.

Con esto, tendrás un proyecto integral que demuestra todos los resultados de aprendizaje de la Unidad 2, listo para que los estudiantes usen las herramientas de IA guiados por tus requerimientos técnicos precisos.

Para un entorno de contenedores, "desconectar el cable" se vuelve virtual y tenemos métodos mucho más quirúrgicos y educativos para demostrar el `ROLLBACK`.

Aquí tienes tres formas prácticas de provocar esta falla intencional en Docker durante la clase:

**1. Simulación por Código (La más didáctica)**
Inyecta un error manual en Go justo en el medio de la transacción SQL. Esto permite a los estudiantes ver el flujo exacto en el código fuente.

* **Cómo hacerlo:** En la función del repositorio que maneja el checkout, después de ejecutar el `INSERT` del pedido principal y antes de hacer el `COMMIT`, añade un error forzado:
```go
tx.Exec("INSERT INTO pedidos...") // Paso 1 exitoso

// Inyección del error para la clase:
return errors.New("Falla simulada: Caída del sistema antes de procesar los items")

tx.Exec("INSERT INTO pedido_items...") // Nunca llega aquí
tx.Commit() 

```


* **El resultado:** El frontend recibe el HTTP 500. Al revisar PostgreSQL (vía DBeaver o la terminal de Docker), los estudiantes verán que el pedido principal no existe, demostrando que el `defer tx.Rollback()` hizo su trabajo.

**2. Violación de Integridad Relacional (Falla de Datos)**
Fuerza a la base de datos a abortar la transacción enviando datos inválidos que rompan las reglas (Constraints), demostrando la robustez de PostgreSQL.

* **Cómo hacerlo:** Desde Postman o modificando temporalmente la petición en React, envía un carrito que contenga un producto con un `id` que no exista en la tabla de productos (ej. `id: 99999`).
* **El resultado:** El primer `INSERT` (el pedido) funciona en memoria, pero el segundo `INSERT` (los items) viola la clave foránea (`foreign key constraint`). PostgreSQL aborta automáticamente la transacción. React muestra error y la base de datos queda limpia.

**3. Caos a nivel de Contenedor (El "Cable de Red" virtual)**
Esta es la simulación más dramática y realista de una caída de infraestructura a mitad de un proceso de misión crítica.

* **Cómo hacerlo:** Añade una pausa temporal en el código de Go justo antes del `COMMIT`:
```go
time.Sleep(10 * time.Second) // Pausa dramática para la clase
tx.Commit()

```


Haz clic en "Confirmar Pedido" en React. Mientras el navegador está esperando (cargando), ve a tu terminal y detén bruscamente el contenedor de la base de datos:
`docker kill multicatalogo-db`
* **El resultado:** La conexión se corta abruptamente. Cuando vuelvas a levantar el contenedor (`docker start multicatalogo-db`), la base de datos aplicará su propio mecanismo de recuperación y el pedido a medio hacer habrá desaparecido, demostrando las propiedades ACID a nivel de motor.

¿Te gustaría que estructuremos el fragmento de código exacto en Go con el bloque `tx.Begin()`, el manejo del `defer tx.Rollback()` y el punto exacto de inyección del error para tu clase?